In [ ]:
# Imports
import math
import os
import pickle
import random
import time
from collections import defaultdict
from typing import Type

import numpy as np
import pandas as pd
import pulp

# Imports
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager

In [ ]:
# Load data
df = pd.read_csv("scraped_data_with_penalty.csv")

In [ ]:
md_inds = df[df["car_id"].str.startswith("94/Porsche 935 'Moby Dick'/1978")].index
df.loc[md_inds, "tag_Year_of_the_Horsepower"] = 0

In [ ]:
# Add all makes
makes = pd.get_dummies(
    df["make"],
    columns=["make"],
    prefix="make",
    dtype=int,
)
df = pd.concat([df, makes], axis=1)

In [ ]:
def remove_non_owned(df: pd.DataFrame, upgraded: bool) -> pd.DataFrame:
    penalties = {80: 75000, 65: 10000, 50: 1650, 40: 425, 30: 175, 20: 55, 0: 85}
    # Initialise new df
    new_df = pd.DataFrame(columns=df.columns)

    # Iterate through rq ranges and add owned cars to new df
    rq_upper = 150
    for rq_lower, non_owned_pen in penalties.items():
        # Mask df to the current rq range
        mask = (df["rq"] >= rq_lower) & (df["rq"] < rq_upper)
        masked_df = df[mask]

        # Get cars with less than the penalty
        if upgraded:
            owned_cars = masked_df[masked_df["penalty"] == 0]
        else:
            owned_cars = masked_df[masked_df["penalty"] < non_owned_pen]

        # Add owned cars to new df
        if rq_upper == 150:
            new_df = owned_cars.copy()
        else:
            new_df = pd.concat([new_df, owned_cars], ignore_index=True)

        # Update rq_upper for next iteration
        rq_upper = rq_lower

    return new_df.reset_index(drop=True)

In [ ]:
load_challenge = True

In [ ]:
challenge_cats = ["Skyline", "YotH"]
challenge_cat_num = 1
challenge_num = 9
challenge_cat = challenge_cats[challenge_cat_num]

# Add necessary columns and non-owned cars
challenges_setup_dict = {
    "Skyline": {
        "base": {"sr": 1, "er": 10, "useful": df["tag_Japan_Pro_Tour"]},
        4: {"sr": 8, "er": 10, "useful": (df["year"] >= 2010) & (df["rq"].between(50, 64))},
    },
    "YotH": {
        "base": {"sr": 1, "er": 12, "useful": df["tag_Year_of_the_Horsepower"]},
        4: {"sr": 8, "er": 12, "useful": False},
        5: {"useful": df["rq"] >= 50},
        6: {"useful": df["rq"] >= 50},
        7: {"useful": df["rarity_F"]},
        8: {"useful": df["rarity_E"]},
        9: {"er": 3, "useful": df["rarity_D"]},
        10: {"useful": df["rarity_C"]},
        11: {"useful": df["rarity_B"]},
        12: {"useful": df["rq"].between(65, 115)},
    },
}

# Get indiv dicts and base mask
setup_dict = challenges_setup_dict[challenge_cat].get(challenge_num, {})
base_dict = challenges_setup_dict[challenge_cat]["base"]
base_mask = base_dict["useful"]

# Check for different details
sr = setup_dict.get("sr", base_dict["sr"])
er = setup_dict.get("er", base_dict["er"])
# er = sr
useful_mask = ((df["rq"] > 0) & base_mask & setup_dict.get("useful", True)).astype(bool)
# useful_mask = ((df['rq'] > 0) & False).astype(bool)
useful_df = df[useful_mask]
exclude_rounds = setup_dict.get("exc", [])

challenges = {
    "Skyline": {
        "base": ["Skyline Nismo ", 1, 10, {"tag_Japan_Pro_Tour": [((1, 10), 5)]}],
        4: [
            "4",
            8,
            0,
            {
                "rarity_D": [((1, 1), 1), ((2, 2), 2), ((3, 3), 3)],
                "rarity_C": [((4, 4), 2), ((5, 5), 3), ((6, 6), 4), ((7, 7), 5)],
                "rarity_B": [((8, 8), 3), ((9, 9), 4), ((10, 10), 5)],
                "year range 2010 2030": [((1, 10), 5)],
            },
        ],
        6: [
            "6",
            0,
            0,
            {
                "rarity_D": [((1, 1), 3), ((2, 2), 4), ((3, 3), 5)],
                "rarity_C": [((4, 4), 2), ((5, 5), 3), ((6, 6), 4), ((7, 7), 5)],
                "rarity_B": [((8, 8), 1), ((9, 9), 2), ((10, 10), 3)],
                "year range 1980 1999": [((1, 10), 5)],
            },
        ],
    },
    "YotH": {
        "base": ["Horsepower: ", 1, 12, {"tag_Year_of_the_Horsepower": [((1, 12), 5)]}],
        4: ["National", 0, 0, {}],
        5: ["Mounted", 0, 0, {}],
        6: [
            "Uphill",
            0,
            0,
            {
                "rarity_F": [((1, 6), 1)],
                "rarity_E": [((1, 12), 1)],
                "rarity_D": [((1, 12), 1)],
                "rarity_C": [((1, 12), 1)],
                "rarity_B": [((1, 12), 1)],
                "rarity_A": [((7, 12), 1)],
            },
        ],
        7: ["Spirited", 0, 0, {"RQ_range_10_": [((1, 6), 5)], "rarity_F": [((1, 12), 5)]}],
        8: ["Rearing", 0, 0, {"RQ_range_20_25": [((1, 6), 5)], "rarity_E": [((1, 12), 5)]}],
        9: ["Blazing", 0, 0, {"RQ_range_30_34": [((1, 6), 5)], "rarity_D": [((1, 12), 5)]}],
        10: ["Galloping", 0, 0, {"RQ_range_40_": [((1, 6), 5)], "rarity_C": [((1, 12), 5)]}],
        11: ["Wild", 0, 0, {"RQ_range_50_": [((1, 6), 5)], "rarity_B": [((1, 12), 5)]}],
        12: ["Bucking", 0, 0, {"RQ_range_65_115": [((1, 12), 5)]}],
    },
}

In [ ]:
# Get the base and specific details
base_name_start, base_sr, base_er, base_restrictions = challenges[challenge_cat]["base"]
num_name_start, num_sr, num_er, num_restrictions = challenges[challenge_cat][challenge_num]

# Combine
challenge_name_start = base_name_start + num_name_start
start_round = base_sr if num_sr == 0 else num_sr
end_round = base_er if num_er == 0 else num_er
challenge_restrictions = base_restrictions | num_restrictions

In [ ]:
if useful_df.empty:
    # Only owned cars allowed?
    only_owned = True
    # Only upgraded cars allowed?
    only_upgraded = False
    # If only owned, remove all non-owned cars
    if only_owned:
        df = remove_non_owned(df, upgraded=only_upgraded)
else:
    car_versions = 2
    useful_df = useful_df[useful_df["car_version"] <= car_versions - 1]
    owned_df = remove_non_owned(df, upgraded=False)
    df = pd.concat([owned_df, useful_df], ignore_index=True).drop_duplicates()

In [ ]:
# Set max penalty
max_penalty = 30000
# Remove all cars with penalty higher than this value
df = df[df["penalty"] <= max_penalty]

In [ ]:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

In [ ]:
def check_challenge_eligibility(challenge_dict):
    # Iterate through rounds
    for round_key, round_dict in challenge_dict.items():
        # Check round_key is an integer
        if not isinstance(round_key, int):
            raise ValueError(f"round_key key {round_key} is not an integer.")

        # Check RQ limit is an integer greater than 50, less than 500 (inclusive)
        if not isinstance(round_dict["RQ limit"], int):
            raise ValueError(f"round_key {round_key} RQ limit is not an integer.")
        if round_dict["RQ limit"] < 50:
            raise ValueError(f"round_key {round_key} RQ limit is less than 50.")
        if round_dict["RQ limit"] > 500:
            raise ValueError(f"round_key {round_key} RQ limit is greater than 500.")

        # Check restrictions is a dictionary
        if not isinstance(round_dict["Restrictions"], dict):
            raise ValueError(f"round_key {round_key} restrictions is not a dictionary.")

        # Check Tracks is a dictionary
        if not isinstance(round_dict["Tracks"], dict):
            raise ValueError(f"round_key {round_key} tracks is not a dictionary.")

        # Check track keys are 1-5
        track_count = 1
        for track_k in round_dict["Tracks"].keys():
            if track_k != track_count:
                raise ValueError(f"round_key {round_key} track key {track_k} is not in order.")
            track_count += 1

        # Check each track value is a tuple of (track name, distance)
        track_names = df.columns.tolist()
        for track_name, track_time in round_dict["Tracks"].values():
            if track_name not in track_names:
                raise ValueError(
                    f"round_key {round_key} track name {track_name} is not in dataframe columns."
                )
            if not isinstance(track_time, (float, int)):
                raise ValueError(f"round_key {round_key} track time {track_time} is not a float.")
            if track_time < 0:
                raise ValueError(f"round_key {round_key} track time {track_time} is negative.")

    return True

In [ ]:
def convert_to_seconds(time_str):
    try:
        time_in_seconds = 0
        time_in_seconds += float(time_str.split(":")[0]) * 60  # minutes to seconds
        time_in_seconds += float(time_str.split(":")[1])  # seconds
        time_in_seconds += float(time_str.split(":")[2]) / 100  # deci-
        time_in_seconds = round(time_in_seconds, 2)
        return round(float(time_in_seconds), 2)
    except:
        return np.inf

In [ ]:
track_extra_map = {"roll": " (R)", "clearance": " (C)"}
condition_map = {
    "Type_00": "Dry",
    "Type_01": "Wet",
    "Type_10": "Dirt",
    "Type_20": "Gravel",
    "Type_50": "Sand",
    "Type_60": "Snow",
    "Type_11": "Dirt Wet",
    "Type_30": "Ice",
    "Type_41": "Aspht Dirt Wet",
    "Type_c0": "Aspht Sand",
    "Type_40": "Aspht Dirt",
    "Type_i0": "Aspht",
    "Type_e0": "Sand Dirt",
    "Type_f0": "Aspht Grass Dirt",
    "Type_c1": "Aspht Sand Wet",
    "Type_70": "Grass",
    "Type_b0": "Aspht Gravel",
    "Type_d0": "Aspht Snow",
    "Type_71": "Grass Wet",
    "Type_h1": "Snow Dirt Wet",
    "Type_51": "Sand Wet",
    "Type_m0": "Aspht Dirt",
    "Type_k0": "Dirt",
    "Type_g0": "Ice Snow",
}

In [ ]:
def scrape_challenge(
    challenge_name_start: str, start_round: int, end_round: int, challenge_restrictions: dict
):
    # set up driver
    with open("device_details.pkl", "rb") as f:
        device_details = pickle.load(f)
        window_width = device_details["width"]
        window_height = device_details["height"]
    service = Service(GeckoDriverManager().install())
    options = webdriver.FirefoxOptions()
    driver = webdriver.Firefox(service=service, options=options)
    driver.set_window_rect(x=0, y=0, width=window_width, height=window_height)
    driver.execute_script("document.body.style.overflowX = 'hidden';")

    # load challenges home page
    driver.get("https://www.topdrivesrecords.com/challenges")
    time.sleep(2)

    # Initialise dict to store data
    challenge_dict = {}

    # Select correct challenge
    # Identify all challenges
    challnege_container_xpath = "/html/body/div/div[2]/div[1]/div[2]/div/div"
    challenge_container = driver.find_element(By.XPATH, challnege_container_xpath)
    challenge_buttons = challenge_container.find_elements(By.TAG_NAME, "button")
    # Find and click on the correct challenge
    # print(challenge_name_start)
    for challenge_button in challenge_buttons:
        challenge_name = challenge_button.text
        # print(challenge_name)
        if challenge_name.startswith(challenge_name_start):
            challenge_button.click()
            break
    time.sleep(1)

    # Check number of rounds
    # Open round dropdown
    round_dropdown_xpath = "/html/body/div/div[2]/div[1]/div[1]/div/div[2]/div/div[2]/div[2]/button"
    try:
        driver.find_element(By.XPATH, round_dropdown_xpath).click()
    except:
        input()
        driver.find_element(By.XPATH, round_dropdown_xpath).click()
    # Locate all round buttons
    round_buttons_container_xpath = "/html/body/div/div[2]/div[9]/div[2]/div/div/div[2]"
    round_buttons_container = driver.find_element(By.XPATH, round_buttons_container_xpath)
    round_buttons = round_buttons_container.find_elements(By.TAG_NAME, "button")
    # Get number of rounds
    first_round = round_buttons[1]
    first_round_text = first_round.find_element(By.XPATH, "./div[1]").text
    last_round = round_buttons[-1]
    last_round_text = last_round.find_element(By.XPATH, "./div[1]").text
    # Assert start and end rounds are within range
    assert start_round >= int(first_round_text.split(" ")[1]), (
        f"Start round {start_round} is before first round {first_round_text.split(' ')[1]}"
    )
    assert end_round <= int(last_round_text.split(" ")[1]), (
        f"End round {end_round} is after last round {last_round_text.split(' ')[1]}"
    )
    # Select start round
    for round_button in round_buttons[1:]:
        round_text = round_button.find_element(By.XPATH, "./div[1]").text
        if round_text.split(" ")[1] == str(start_round):
            round_button.click()
            break

    # Iterate through rounds
    for round_num in range(start_round, end_round + 1):
        time.sleep(0.5)

        # Initialise round dict
        round_dict = {"Tracks": {}, "Restrictions": {}, "RQ range": [0, 150], "RQ limit": 600}

        # Get RQ limit
        rq_limit_xpath = (
            "/html/body/div/div[2]/div[1]/div[1]/div/div[2]/div/div[2]/div[4]/div[1]/span[4]"
        )
        rq_limit = int(driver.find_element(By.XPATH, rq_limit_xpath).text.strip())
        round_dict["RQ limit"] = rq_limit

        # Add restrictions
        for restriction, value_tuples in challenge_restrictions.items():
            # Find the tuple for the current round
            for value_tuple in value_tuples:
                if value_tuple[0][0] <= round_num <= value_tuple[0][1]:
                    # If rq involved, treat differently
                    if "RQ range" in restriction:
                        round_dict["RQ range"] = [int(rq) for rq in restriction.split(" ")[2:]]
                    else:
                        round_dict["Restrictions"][restriction] = value_tuple[1]

        # Locate track divs
        track_container_xpath = "/html/body/div/div[2]/div[1]/div[2]/div[2]"
        track_container = driver.find_element(By.XPATH, track_container_xpath)
        track_divs = track_container.find_elements(By.XPATH, "./div")

        # Iterate through track divs
        for i, track_div in enumerate(track_divs):
            # Get track name

            track_name_container = track_div.find_element(By.XPATH, "./div[2]/div/div")
            track_name = track_name_container.find_element(By.XPATH, "./div[1]").text

            # Check for R/C
            try:
                track_extra_class = track_name_container.find_element(
                    By.XPATH, "./div[1]/span/i"
                ).get_attribute("class")
                track_extra_key = track_extra_class.split("tdicon-")[1].split(" ")[0]
                if track_extra_key in track_extra_map.keys():
                    track_name += track_extra_map[track_extra_key]
            except NoSuchElementException:
                pass

            # Get conditions
            track_class = track_name_container.get_attribute("class")
            track_classes = track_class.split(" ")
            track_type = next((c for c in track_classes if c.startswith("Type_")), None)
            track_name += " / "
            track_name += condition_map[track_type]

            # Get time for track
            track_time_str = track_div.find_element(By.XPATH, "./div[3]/div/div/div[1]").text
            # If not test bowl, convert time to seconds
            if "Test Bowl" in track_name:
                track_time = int(track_time_str)
            else:
                track_time = convert_to_seconds(track_time_str)

            # Add to round dict
            round_dict["Tracks"][i + 1] = (track_name, track_time)

        # Add round dict to challenge dict
        challenge_dict[round_num] = round_dict

        # If not the last round or end round, click on next round button
        if round_num != end_round:
            next_round_button_xpath = (
                "/html/body/div/div[2]/div[1]/div[1]/div/div[2]/div/div[3]/button"
            )
            driver.find_element(By.XPATH, next_round_button_xpath).click()
            time.sleep(1)

    # Close driver
    driver.quit()

    return challenge_dict

In [ ]:
# Get challenge data

# If veteran, and more than a week old, scrape new
if challenge_cat == "Veteran":
    try:
        if (time.time() - os.path.getmtime("Challenges/Veteran.pkl")) / 86400 > 7:
            print("Veteran challenge old, scraping new")
            load_challenge = False
    except FileNotFoundError:
        print("No existing Veteran challenge")
        load_challenge = False

# Try to load challenge data from pickle file
try:
    if not load_challenge:
        raise FileNotFoundError
    with open(f"Challenges/{challenge_name_start.replace(':', '-')}.pkl", "rb") as f:
        challenge_dict = pickle.load(f)
    print(f"Loaded challenge data from Challenges/{challenge_name_start.replace(':', '-')}.pkl")
    # Check its eligibility
    challenge_eligible = check_challenge_eligibility(challenge_dict)
    if challenge_eligible:
        print(f"Challenge is eligible: {challenge_eligible}")
    else:
        raise ValueError(f"Challenge is not eligible: {challenge_eligible}")
except FileNotFoundError:
    if load_challenge:
        print(
            f"Challenge data not found for file {challenge_name_start.replace(':', '-')}.pkl.",
            end=" ",
        )
    print("Scraping challenge data...")
    challenge_dict = scrape_challenge(
        challenge_name_start, start_round, end_round, challenge_restrictions
    )
    # save challenge_dict
    with open(f"Challenges/{challenge_name_start.replace(':', '-')}.pkl", "wb") as f:
        pickle.dump(challenge_dict, f)

Loaded challenge data from Challenges/Horsepower- Blazing.pkl
Challenge is eligible: True


In [ ]:
challenge_dict

{1: {'Tracks': {1: ('Tokyo Loop (R) / Wet', 39.91),
   2: ('Tokyo G-Force Test / Wet', 67.25),
   3: ('G-Force Test / Wet', 28.3),
   4: ('Car Park / Wet', 53.88),
   5: ('Fast Circuit / Wet', 83.17)},
  'Restrictions': {'tag_Year_of_the_Horsepower': 5,
   'RQ_range_30_34': 5,
   'rarity_D': 5},
  'RQ range': [0, 150],
  'RQ limit': 160},
 2: {'Tracks': {1: ('Slalom Test / Wet', 36.56),
   2: ('Car Park / Wet', 53.88),
   3: ('G-Force Test / Wet', 28.3),
   4: ('City Streets Medium (C) / Wet', 96.77),
   5: ('Tokyo Loop (R) / Wet', 39.91)},
  'Restrictions': {'tag_Year_of_the_Horsepower': 5,
   'RQ_range_30_34': 5,
   'rarity_D': 5},
  'RQ range': [0, 150],
  'RQ limit': 160},
 3: {'Tracks': {1: ('Tokyo Loop (R) / Dry', 36.98),
   2: ('Tokyo Bridge / Dry', 60.23),
   3: ('Twisty Circuit / Dry', 78.26),
   4: ('Tokyo G-Force Test / Dry', 59.73),
   5: ('Tokyo Off-Ramp (R) / Dry', 55.88)},
  'Restrictions': {'tag_Year_of_the_Horsepower': 5,
   'RQ_range_30_34': 5,
   'rarity_D': 5},
  'R

In [ ]:
new_dict = {}
for round_num, round_dict in challenge_dict.items():
    if round_num in exclude_rounds:
        continue
    if sr <= round_num <= er:
        new_dict[round_num] = round_dict

challenge_dict = new_dict

In [ ]:
with open("track_pts_uppers.pkl", "rb") as f:
    track_uppers = pickle.load(f)

In [ ]:
def calc_points(track_time, track_name, time_to_beat):
    # Different function for test bowl
    if "Test Bowl" in track_name:
        # pts = 350 * x - 350
        # x = win / lose
        win = max(track_time, time_to_beat)
        lose = min(track_time, time_to_beat)
        if lose == 0:
            pts = 50
        else:
            x = win / lose
            pts = 350 * x - 350

        # Win/Lose
        if track_time < time_to_beat:
            pts *= -1

    else:
        # Get upper pts limit
        upper = track_uppers[track_name]
        # pts = upper * (-1 / (x + 1) + 1)
        # x = (lose - win) / win
        win = min(track_time, time_to_beat)
        lose = max(track_time, time_to_beat)
        # possibilities:
        #   loser doesn't finish
        #     winner doesn't finish: assume loss - -50pts
        #     winner finishes: 250pts
        #   loser finishes: standard
        if lose > 9999999999:
            if win > 9999999999:
                return -50
            else:
                pts = 250
        else:
            x = (lose - win) / win
            # print(x)
            pts = upper * (-1 / (x + 1) + 1)

        # Win/Lose
        if track_time > time_to_beat:
            pts *= -1

    # Round close results to 50/-50
    if -50 < pts < 0:
        pts = -50
    if 0 < pts < 50:
        pts = 50

    return math.floor(pts)

In [ ]:
for i, col in enumerate(df.columns):
    print(i, col)

0 0-100-0mph / Dry
1 0-100-0mph / Wet
2 0-100-0mph / Dirt
3 0-100mph / Dry
4 0-100mph / Wet
5 0-100mph / Dirt
6 0-100mph / Gravel
7 0-120mph / Dry
8 0-120mph / Gravel
9 0-124mph / Dry
10 0-150-0mph / Dry
11 0-150mph / Dry
12 0-150mph / Wet
13 0-150mph / Dirt
14 0-170mph / Dry
15 0-200mph / Dry
16 0-60-0mph / Gravel
17 0-60mph / Dry
18 0-60mph / Wet
19 0-60mph / Gravel
20 0-60mph / Sand
21 0-62mph / Dry
22 1 Mile Drag / Dry
23 1 Mile Drag / Wet
24 1 Mile Drag / Dirt
25 1 Mile Drag / Gravel
26 1 Mile Drag / Sand
27 1 Mile Drag / Snow
28 1 Mile Drag (R) / Dry
29 1/2 Mile Drag / Dry
30 1/2 Mile Drag / Wet
31 1/2 Mile Drag / Dirt Wet
32 1/2 Mile Drag / Gravel
33 1/2 Mile Drag / Sand
34 1/2 Mile Drag / Snow
35 1/4 Mile Drag / Dry
36 1/4 Mile Drag / Wet
37 1/4 Mile Drag / Dirt Wet
38 1/4 Mile Drag / Gravel
39 1/4 Mile Drag / Ice
40 1/4 Mile Drag / Sand
41 1/4 Mile Drag / Snow
42 1/4 Mile Drag (R) / Dry
43 100-150mph (R) / Dry
44 30-130mph (R) / Dry
45 50-150mph (R) / Dry
46 75-125mph (R) / Dr

In [ ]:
# Create new df with points for each track

# Start with copying direct cols we need
# skip stats
copy_cols = df.columns[346:351].to_list()  # rq, make, model, mm, year
copy_cols += df.columns[354:357].to_list()  # eng, wei, cha
copy_cols += df.columns[360:].to_list()  # tags / filters
challenge_df = df[copy_cols].copy()

# Add track columns
track_cols = []
for round_i, round_dict in challenge_dict.items():
    for track_j, (track_name, track_time) in round_dict["Tracks"].items():
        # Add column to df
        challenge_df[f"{round_i}.{track_j}"] = df[track_name].apply(
            calc_points, args=(track_name, track_time)
        )
        # Add to track cols
        track_cols.append(f"{round_i}.{track_j}")

In [ ]:
# Get a set of all restrictions and add a column if not already in
all_restrictions = set()
all_ranges = set()
for round_i, round_dict in challenge_dict.items():
    for restriction in round_dict["Restrictions"].keys():
        all_restrictions.add(restriction)
        if restriction not in challenge_df.columns:
            print(f"Adding col for {restriction}")
            challenge_df[restriction] = 0
            if restriction.startswith("RQ"):
                _, _, lower, upper = restriction.split("_")
                challenge_df[restriction] = challenge_df["rq"].apply(
                    lambda x: 1 if int(lower) <= x <= int(upper) else 0
                )
    all_ranges.add(tuple(round_dict["RQ range"]))

Adding col for RQ_range_30_34


In [ ]:
all_restrictions

{'RQ_range_30_34', 'rarity_D', 'tag_Year_of_the_Horsepower'}

In [ ]:
# Get a set of universal restrictions
universal_restrictions = all_restrictions.copy()
for round_i, round_dict in challenge_dict.items():
    # Go through all universal restrictions and check if they are in this round
    for univ_restriction in list(universal_restrictions):
        if univ_restriction not in round_dict["Restrictions"].keys():
            universal_restrictions.remove(univ_restriction)
        else:
            if round_dict["Restrictions"][univ_restriction] != 5:
                universal_restrictions.remove(univ_restriction)

In [ ]:
# Make sure any A/B/... restrictions have a column
for restriction in all_restrictions:
    if "/" in restriction:
        # Add column
        challenge_df[restriction] = 0
        # Get all sub-restrictions
        sub_restrictions = restriction.split("/")
        # Check if all sub-restrictions are in the df
        for sub_restriction in sub_restrictions:
            if sub_restriction not in challenge_df.columns:
                challenge_df[sub_restriction] = 0
                raise ValueError(f"Sub-restriction {sub_restriction} not found in df columns.")
        # Set the column to 1 if any of the sub-restrictions are 1
        challenge_df[restriction] = challenge_df[sub_restrictions].max(axis=1)
    # add year range column if needed
    if restriction.startswith("year range"):
        min_year = int(restriction.split(" ")[-2])
        max_year = int(restriction.split(" ")[-1])
        challenge_df[restriction.replace(" ", "_")] = 0
        challenge_df.loc[
            (challenge_df["year"] >= min_year) & (challenge_df["year"] <= max_year),
            restriction.replace(" ", "_"),
        ] = 1

In [ ]:
# add year range column if needed
for restriction in all_restrictions:
    if restriction.startswith("year range"):
        min_year = int(restriction.split(" ")[-2])
        max_year = int(restriction.split(" ")[-1])
        challenge_df[restriction.replace(" ", "_")] = 0
        challenge_df.loc[
            (challenge_df["year"] >= min_year) & (challenge_df["year"] <= max_year),
            restriction.replace(" ", "_"),
        ] = 1

In [ ]:
all_restrictions

{'RQ_range_30_34', 'rarity_D', 'tag_Year_of_the_Horsepower'}

In [ ]:
# add rq range column if needed
for rq_range in list(all_ranges):
    if rq_range == (0, 150):
        continue
    rq_lower = rq_range[0]
    rq_upper = rq_range[1]
    challenge_df[f"RQ_range_{rq_lower}_{rq_upper}"] = 0
    challenge_df.loc[
        (challenge_df["rq"] >= rq_lower) & (challenge_df["rq"] <= rq_upper),
        f"RQ_range_{rq_lower}_{rq_upper}",
    ] = 1

In [ ]:
all_restrictions

{'RQ_range_30_34', 'rarity_D', 'tag_Year_of_the_Horsepower'}

# make sure all restrictions have their own column
for restriction in all_restrictions:
    if 'year range' in restriction:
        # Get range
        start_year = int(restriction.split(' ')[2])
        end_year = int(restriction.split(' ')[3])
        # Get all cars that meet the restriction
        challenge_df[restriction.replace(' ', '_')] = challenge_df['year'].apply(lambda x: 1 if start_year <= x <= end_year else 0)
    elif 'make' in restriction:
        make = restriction.split('make_')[1]
        # Get all cars that meet the restriction
        challenge_df[restriction] = challenge_df['make'].apply(lambda x: 1 if x == make else 0)


In [ ]:
# If any restrictions require 5x for every round, delete all cars not meeting this restriction
for restriction in universal_restrictions:
    challenge_df = challenge_df[challenge_df[restriction.replace(" ", "_")] == 1]

In [ ]:
# Add winning cars for round 10.5

if challenge_cat == "Filberto" and challenge_num == 7:
    cars = [
        ("GMC Safari", [(3, 3, 2), (3, 2, 3)]),
        ("Lincoln Mark III", [(3, 3, 2), (3, 2, 3), (2, 3, 3)]),
        ("Peugeot 1007 RC Concept", [(3, 3, 2), (3, 2, 3), (2, 3, 3)]),
        ("Chevrolet Corvair Monza", [(3, 3, 2), (3, 2, 3), (2, 3, 3)]),
        ("Fiat Doblo Cargo 1.6 Multijet II", [(3, 3, 2)]),
    ]
    for i in challenge_df.index:
        car_row = challenge_df.loc[i]
        for car in cars:
            if car_row["make_model"] == car[0]:
                for ups in car[1]:
                    if car_row["engine_up"] == ups[0] and car_row["weight_up"] == ups[1]:
                        challenge_df.loc[i, "10.5"] = 50

In [ ]:
# Remove all rows which lose all tracks
challenge_df = challenge_df[~(challenge_df[track_cols].max(axis=1) < 0)]

In [ ]:
for col in challenge_df.select_dtypes(include="int"):
    # if 0 or less, print
    if challenge_df[col].max() <= 0:
        print(col)

prize_Yes
country_AE
country_AR
country_AT
country_AU
country_BR
country_CH
country_CN
country_DE
country_DK
country_HR
country_IN
country_MX
country_MY
country_NL
country_NZ
country_ZA
tyres_Off-road
tyres_Slick
fuel_Bioethanol
fuel_Electric
fuel_Hybrid
fuel_Hydrogen
fuel_Misc
seats_1
seats_6
seats_7
seats_8
seats_9
seats_11
engine_pos_Mid
engine_pos_Mid-rear
engine_pos_Mixed
engine_pos_Rear
tag_Purple
tag_Eco_Friendly
tag_None
tag_Style_Icon
tag_Ministry_of_Racing:_Crown_Pursuit
tag_Enter_the_Black_Forest
tag_French_Riviera
tag_Boss_Rush_Collection
tag_German_Renaissance
tag_Loch_to_Loch
tag_Drivers_Choice
tag_Brown
tag_Beige
tag_European_Revolution
tag_Ximena's_Collection_2
tag_Orange
tag_Turquoise
tag_European_Grand_Tour
tag_Track
tag_Pink
tag_Muscle_Car
tag_Touma's_Collection_2
tag_Learn_the_Savannah_Way
tag_German_Powerhaus
tag_Autobahn_Icons
tag_Filberto's_Collection_2
tag_Amalfi_Coast_Cruising
tag_Crown_Pursuit
tag_Asia-Pacific_Revival
tag_Yellow
tag_Motorsport
tag_Street_Racer

# Set columns to all 1s for specific challenges
columns_to_set = []

if challenge_num == 7:
    columns_to_set.append('2.5')

for col in columns_to_set:
    if col in challenge_df.columns:
        challenge_df[col] = 1
    else:
        print(f'Column {col} not found in df columns.')

In [ ]:
# Convert to dictionary
data_dict = challenge_df.to_dict(orient="index")

In [ ]:
# Create the problem
problem = pulp.LpProblem("Challenge", pulp.LpMinimize)

In [ ]:
# Define variables

# Get the car IDs and track IDs
car_keys = list(data_dict.keys())
track_ids = [f"{round_key}.{i + 1}" for round_key in challenge_dict.keys() for i in range(5)]
# One variable for each car and track combination
x = pulp.LpVariable.dicts(
    "car_track_use", [(i, t) for i in car_keys for t in track_ids], cat="Binary"
)

In [ ]:
# Define objective function

# Minimise total penalty for the challenge
y = pulp.LpVariable.dicts(
    "car_used", car_keys, cat="Binary"
)  # Binary variable for each car used (so a penalised car used more than once only gets one penalty)
objective = pulp.lpSum([data_dict[i]["penalty"] * y[i] for i in car_keys])
problem += objective, "Total_Penalty"

In [ ]:
# For all car ids get their indexes
car_id_groups = defaultdict(list)
for i in car_keys:
    car_id = data_dict[i]["car_id"]
    car_id_groups[car_id].append(i)

In [ ]:
# Add constraints

# Constraint 0: If a car is used, update y
# For each car
for i in car_keys:
    # ... update y if the car is used in any track
    problem += pulp.lpSum([x[(i, t)] for t in track_ids]) <= len(track_ids) * y[i], f"Link_x_y_{i}"

# Constraint 1: Each race must have exactly one assigned car
# For all tracks...
for t in track_ids:
    # ... exactly one car must be assigned.
    problem += pulp.lpSum([x[(i, t)] for i in car_keys]) == 1, f"One_Car_Per_Track_{t}"

# Constraint 2: Each car can only be used once per round
# For each round...
for round_key in challenge_dict.keys():
    # ... for each car...
    for i in car_keys:
        # ... the sum of all tracks in that round must be less than or equal to 1 (0 or 1).
        problem += (
            pulp.lpSum([x[(i, t)] for t in track_ids if t.startswith(f"{round_key}.")]) <= 1,
            f"One_Car_Use_Per_Round_{round_key}_{i}",
        )

# Constraint 3: Each round must have a total score of 250
# For each round...
for round_key, round_info in challenge_dict.items():
    # ... the sum of all cars' success scores in that round must be equal to 5.
    problem += (
        pulp.lpSum(
            data_dict[i][t] * x[(i, t)]
            for i in car_keys
            for t in track_ids
            if t.startswith(f"{round_key}.")
        )
        >= 250,
        f"3_star_per_round_{round_key}",
    )

# Constraint 4: Each round must not exceed RQ limit
# For each round...
for round_key, round_info in challenge_dict.items():
    # ... the sum of all cars' rq in that round must be less than or equal to the RQ limit.
    rq_limit = round_info["RQ limit"]
    problem += (
        pulp.lpSum(
            data_dict[i]["rq"] * x[(i, t)]
            for i in car_keys
            for t in track_ids
            if t.startswith(f"{round_key}.")
        )
        <= rq_limit,
        f"RQ_Limit_{round_key}",
    )

# Constraint 5: Each round must have correct restrictions
# For each round...
for round_key, round_info in challenge_dict.items():
    # ... for each restriction...
    for req, quant in round_info["Restrictions"].items():
        # ... the sum of all cars' success scores in that round times the restriction must be greater than or equal to the restriction quantity.
        problem += (
            pulp.lpSum(
                data_dict[i][f"{req.replace(' ', '_')}"] * x[(i, t)]
                for i in car_keys
                for t in track_ids
                if t.startswith(f"{round_key}.")
            )
            >= quant,
            f"Restriction_{round_key}_{req}_{quant}",
        )

# Constraint 6: No duplicate car ids selected
# For each car id group...
for car_id, car_indexes in car_id_groups.items():
    # ... only one car from the group can be used in the challenge.
    problem += pulp.lpSum([y[i] for i in car_indexes]) <= 1, f"One_Car_Per_Car_ID_{car_id}"

In [ ]:
def solve_and_print_results(
    problem: Type[pulp.LpProblem],
    car_data: dict,
    challenge_dict: dict,
    challenges: dict,
    challenge_cat: str,
    challenge_num: int,
    time_limit: int = 600,
) -> None:
    colour_ranges = [
        (0, 19, "240"),
        (20, 29, "34"),
        (30, 39, "27"),
        (40, 49, "226"),
        (50, 64, "196"),
        (65, 79, "129"),
        (80, 150, "214"),
    ]
    challenge_name = challenges[challenge_cat][challenge_num][0]
    print(f"Solving challenge: {challenge_name} (Challenge {challenge_num})")

    # Solve with msg = 1
    problem.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=time_limit))

    # Print the results
    print("Status:", pulp.LpStatus[problem.status])
    obj_val = pulp.value(problem.objective)
    print("Objective value:", obj_val)

    # Initialise list and set to store the results
    results = []
    cars_with_penalty = set()
    cars_used = set()

    if pulp.LpStatus[problem.status] == "Optimal":
        for round_key, round_info in challenge_dict.items():
            print(f"Round {round_key}:")
            round_reqs = list(round_info["Restrictions"].keys())
            data_cols = ["engine_up", "weight_up", "chassis_up", "penalty", "car_version"]
            round_df = pd.DataFrame(
                columns=[
                    "Index",
                    "Year",
                    "RQ",
                    "Make",
                    "Model",
                    "Track",
                    "Challenge Time",
                    "Track Time",
                    "Points",
                    "Engine Up",
                    "Weight Up",
                    "Chassis Up",
                    "Penalty",
                    "Version",
                ]
                + round_reqs
            )
            for race, track in round_info["Tracks"].items():
                # Initialise round_list
                round_list = [track[0], track[1]]  # [Track name, Challenge time]
                # find the car assigned to this track
                for x_tup in x:
                    if x_tup[1] != f"{round_key}.{race}":
                        continue
                    if pulp.value(x[x_tup]) == 1:
                        car_index = x_tup[0]
                        round_list.insert(
                            0, car_index
                        )  # += Index -> [Index, Track name, Challenge time]
                        round_list.insert(
                            1, car_data[car_index]["year"]
                        )  # += Year -> [Index, Year, Track name, Challenge time]
                        round_list.insert(
                            2, car_data[car_index]["rq"]
                        )  # += RQ -> [Index, Year, RQ, Track name, Challenge time]
                        round_list.insert(
                            3, car_data[car_index]["make"]
                        )  # += Make -> [..., RQ, Make, Track name, Challenge time]
                        round_list.insert(
                            4, car_data[car_index]["model"]
                        )  # += Model -> [..., Make, Model, Track name, Challenge time]
                        round_list.append(
                            df.loc[car_index][track[0]]
                        )  # += Track time -> [..., Challenge time, Track Time]
                        round_list.append(
                            car_data[car_index][f"{round_key}.{race}"]
                        )  # += Points -> [..., Track Time, Points]
                        for col in data_cols:
                            round_list.append(car_data[car_index][col])  # [..., Points, Data cols]
                        # Add the requirements to the round list
                        for req in round_reqs:
                            round_list.append(
                                car_data[car_index][f"{req.replace(' ', '_')}"]
                            )  # [..., Data cols, Requirement cols]
                        break
                round_df.loc[len(round_df)] = round_list
            # Add the round_df to the results list
            results.append(round_df)
            # Print the round_df
            print(round_df)
            total_rq = round_df["RQ"].sum()
            total_pen = round_df["Penalty"].sum()
            total_pts = round_df["Points"].sum()
            print(
                f"{total_pts} pts. Penalty: {int(total_pen)}. RQ Used: {total_rq} / {round_info['RQ limit']}"
            )
            print()
            # Check if any cars have a penalty
            for car_index in round_df["Index"]:
                if round_df.loc[round_df["Index"] == car_index, "Penalty"].values[0] > 0:
                    cars_with_penalty.add(car_index)
            # Add the cars used to the set
            for car_index in round_df["Index"]:
                cars_used.add(car_index)
        # Order the cars first by rq (desc), then alphabetically by make
        cars_used = sorted(
            cars_used, key=lambda x: (car_data[x]["rq"], car_data[x]["make"]), reverse=True
        )
        cars_with_penalty = sorted(
            cars_with_penalty, key=lambda x: (car_data[x]["rq"], car_data[x]["make"]), reverse=True
        )
        # Print the cars with penalty
        print("Cars used:")
        cars_used_list = []
        for car_index in cars_used:
            year = car_data[car_index]["year"]
            rq = car_data[car_index]["rq"]
            colour = next(c for low, high, c in colour_ranges if low <= rq <= high)
            make_model = car_data[car_index]["make_model"]
            version = car_data[car_index]["car_version"]
            penalty = car_data[car_index]["penalty"]
            print(
                f"{year} \033[48;5;{colour}m[{rq}]\033[0m {make_model} (Version {version}): {penalty}"
            )
            cars_used_list.append((rq, make_model, version, penalty))
        print()
        if len(cars_with_penalty) == 0:
            print("No cars with penalty.")
        else:
            print("Cars with penalty:")
            for car_index in cars_with_penalty:
                year = car_data[car_index]["year"]
                rq = car_data[car_index]["rq"]
                colour = next(c for low, high, c in colour_ranges if low <= rq <= high)
                make = car_data[car_index]["make"]
                model = car_data[car_index]["model"]
                version = car_data[car_index]["car_version"]
                ups = f"{car_data[car_index]['engine_up']}{car_data[car_index]['weight_up']}{car_data[car_index]['chassis_up']}"
                penalty = car_data[car_index]["penalty"]
                print(
                    f"{year} \033[48;5;{colour}m[{rq}]\033[0m {make} {model} ({version}): {penalty} - {ups}"
                )

        return True, obj_val, results, cars_used_list
    else:
        return False, None, None, None

In [ ]:
def solve_problem(
    problem: Type[pulp.LpProblem], car_data: dict, challenge_dict: dict, time_limit: int = 600
):
    cars_used = set()

    # Solve the problem
    rand_seed = str(random.randint(1, 999999999))
    problem.solve(
        pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit, options=["-randomSeed", rand_seed])
    )

    # Get results
    problem_status = pulp.LpStatus[problem.status]
    obj_val = pulp.value(problem.objective)
    if problem_status == "Optimal":
        # Extract a list of the cars used, make sure to get their rq, makemodel, version, and penalty
        for round_key, round_info in challenge_dict.items():
            for race, _ in round_info["Tracks"].items():
                for x_tup in x:
                    # Find the car assigned to this track
                    if x_tup[1] != f"{round_key}.{race}":
                        continue
                    if pulp.value(x[x_tup]) == 1:
                        car_index = x_tup[0]
                        year = car_data[car_index]["year"]
                        rq = car_data[car_index]["rq"]
                        make_model = car_data[car_index]["make_model"]
                        version = car_data[car_index]["car_version"]
                        penalty = car_data[car_index]["penalty"]
                        ups = f"{car_data[car_index]['engine_up']}{car_data[car_index]['weight_up']}{car_data[car_index]['chassis_up']}"
                        cars_used.add((year, rq, make_model, version, penalty, ups))
        return True, obj_val, cars_used
    else:
        return False, None, None

In [ ]:
def get_all_matching_indices(cars_used, data_dict):
    indices = []
    for car in cars_used:
        for idx, v in data_dict.items():
            if (v["year"], v["rq"], v["make_model"], v["car_version"]) == car:
                indices.append(idx)
    return indices

# Skyline
## Pro Tour
### 2010s-20s (4229)
- 2021 [62] Subaru Levorg (VN) (0): 302.0 - 323
- 2015 [56] Nissan 370Z Nismo (0): 308.0 - 233
- 2016 [56] Infiniti Q70 Hybrid (Y51) (0): 3308.0 - 323
- 2014 [53] Subaru Forester tS 300 (SJ) (0): 311.0 - 332

### 1980s-90s (609)
- 1996 [61] Subaru Impreza WRX (GC8D) (0): 303.0 - 233
- 1999 [58] Subaru Impreza WRX (GC8G) (0): 306.0 - 323

# YotH

#### 12500 / 16000 / 24000

### 4 (30) - 1500
- 2014 [49] Mazda CX-9 3.7 V6 AWD (0): 20.0 - 323
- 2002 [49] Alfa Romeo 156 GTA (0): 10.0 - 323

### 5 (2567) - 2500
- 2014 [67] Audi S4 Avant (B8) (0): 762.0 - 323
- 2014 [64] Morgan Plus 8 Speedster (0): 1650.0 - 233
- 2014 [59] Audi S1 (0): 155.0 - 323

### 6 (2618) - 3000
- 2002 [77] Acura DN-X (0): 752.0 - 233
- 2014 [64] Morgan Plus 8 Speedster (0): 1650.0 - 332
- 2014 [59] Audi S1 (0): 155.0 - 323
- 2014 [35] Mazda 6 Estate 2.5 (0): 29.0 - 323
- 1990 [32] Nissan Primera eGT (0): 32.0 - 233

### 9 (270) - 4000
- 2002 [36] Dodge Neon ACR (0): 28.0 - 323
- 2014 [35] Mazda 6 Estate 2.5 (0): 29.0 - 323
- 1990 [35] Acura Legend 3.2 V6 (0): 29.0 - 323
- 2014 [34] Hyundai Aslan 3.3 (0): 30.0 - 323
- 1990 [32] Nissan Primera eGT (1): 57.0 - 323
- 1990 [32] Nissan Primera eGT (0): 32.0 - 323
- 1990 [32] Citroen Activa 2 (0): 32.0 - 323
- 1966 [31] Lamborghini 400 GT (0): 33.0 - 332

### 10 (128) - 4000
- 2002 [49] Alfa Romeo 156 GTA (0): 20.0 - 323
- 2002 [45] Volvo S80 T6 (0): 54.0 - 323
- 2014 [45] Volkswagen Polo GTI (0): 54.0 - 323

### 11 (10246) - 4500
- 2014 [64] Morgan Plus 8 Speedster (0): 1650.0 - 233
- 1990 [63] Porsche 911 Turbo (0): 1651.0 - 233
- 2014 [59] Volvo XC60 T6 AWD (0): 155.0 - 323
- 1990 [57] Nissan Pulsar GTI-R (N14) (0): 1657.0 - 323
- 2014 [57] Ford Focus ST Estate Mountune (0): 1657.0 - 233
- 2002 [57] BMW 760i (0): 1657.0 - 323
- 2002 [56] Subaru Legacy STI S401 (BE) (0): 1658.0 - 323
- 2014 [53] Subaru Forester tS 300 (SJ) (0): 161.0 - 332

### 12 (33028) - 8000
- 2002 [77] Acura DN-X (1): 10002.0 - 111
- 2002 [77] Acura DN-X (0): 752.0 - 233
- 2014 [76] BMW i8 (0): 10753.0 - 233
- 2014 [72] Lotus Exige S LF1 (0): 10757.0 - 323
- 2014 [65] Infiniti QX70 (0): 764.0 - 233

In [ ]:
success, objective_value, results, cars_used = solve_and_print_results(
    problem, data_dict, challenge_dict, challenges, challenge_cat, challenge_num, time_limit=300
)

Solving challenge: Blazing (Challenge 9)
Status: Optimal
Objective value: 6.0
Round 1:
   Index  Year  RQ     Make                 Model                     Track  Challenge Time  Track Time  Points  Engine Up  Weight Up  Chassis Up  Penalty  Version  tag_Year_of_the_Horsepower  RQ_range_30_34  rarity_D
0   3576  2002  30  Mercury  Marauder Convertible      Tokyo Loop (R) / Wet           39.91       39.65      50          3          2           3        1        1                           1               1         1
1   3588  1990  32  Citroen              Activa 2  Tokyo G-Force Test / Wet           67.25       63.34      63          3          2           3        1        1                           1               1         1
2   2379  1990  32  Citroen              Activa 2        G-Force Test / Wet           28.30       26.41      73          3          2           3        1        0                           1               1         1
3   3420  2002  30  Mercury  Marauder Con

In [ ]:
raise ValueError

ValueError: 

In [ ]:
# Find all solutions under a certain penalty
obj_vals = set()
car_use_counts = dict()
prev_obj_val = np.inf
break_on_first_increase = False
break_on_second_increase = False
solutions = []
break_count = 100
max_obj = 5000
break_obj_val = max(objective_value, max_obj) if objective_value else max_obj
colour_ranges = [
    (0, 19, "240"),
    (20, 29, "34"),
    (30, 39, "27"),
    (40, 49, "226"),
    (50, 64, "196"),
    (65, 79, "129"),
    (80, 150, "214"),
]
while True:
    # Solve the problem
    challenge_name = challenges[challenge_cat][challenge_num][0]
    success, obj_val, cars_used_set = solve_problem(
        problem, data_dict, challenge_dict, time_limit=600
    )
    obj_vals.add(obj_val)

    # If no solution, print message and break
    if not success:
        # If first run, print message and break, otherwise break
        if len(solutions) == 0:
            print(
                f"No solution found for challenge {challenge_name} ({challenge_num}) under the current restrictions."
            )
            break
        else:
            print("No more solutions.")
            break

    # Check break criteria

    # Check obj val is below max
    if obj_val > break_obj_val:
        print(f"Objective value {obj_val} exceeds break value {break_obj_val}. Stopping search.")
        break
    # Check if objective value has increased
    if break_on_first_increase and len(obj_vals) == 2:
        print("Increase in total penalty, stopping.")
        break
    if break_on_second_increase and len(obj_vals) == 3:
        print("Second increase in total penalty, stopping.")
        break

    # If solution found, print details
    cars_used_list = list(cars_used_set)
    cars_used_list.sort(
        key=lambda x: (x[4], -x[1], x[2])
    )  # Sort by penalty, then by rq, then by make_model
    solutions.append((obj_val, cars_used_list))
    print(f"Solution {len(solutions)}: {obj_val}")
    for car in cars_used_list:
        if car[4] > 0:
            colour = next(c for low, high, c in colour_ranges if low <= car[1] <= high)
            car_str = f"{car[0]} \033[48;5;{colour}m[{car[1]}]\033[0m {car[2]} (V{car[3]} - {car[5]}): {car[4]}"
            print(f"    {car_str}")
            if (car[0], car[1], car[2], car[3]) in car_use_counts:
                car_use_counts[(car[0], car[1], car[2], car[3])]["count"] += 1
            else:
                car_use_counts[(car[0], car[1], car[2], car[3])] = {"tup": car, "count": 1}
    print()

    # If solution has penalty 0, stop
    if obj_val == 0:
        print("Found a solution with no penalty! Stopping search.")
        break

    # Check if enough solutions found
    if len(solutions) >= break_count:
        break

    # Get all indices of the cars used
    cars_used_no_pen = []
    for car in cars_used_list:
        # If car has a penalty, add it to the list
        if car[4] > 0:
            no_pen = (car[0], car[1], car[2], car[3])
            cars_used_no_pen.append(no_pen)
    indices_to_exclude = get_all_matching_indices(cars_used_no_pen, data_dict)

    # Add new constraint excluding the current solution
    if indices_to_exclude:
        # The sum of cars used in the indices to exclude must be less than the number of cars in cars_used_no_pen
        problem += (
            pulp.lpSum([y[i] for i in indices_to_exclude]) <= len(cars_used_no_pen) - 1,
            f"Exclude_Exact_Set_{len(solutions)}",
        )

    prev_obj_val = obj_val

In [ ]:
unowned_values = [
    (0, 19, 170),
    (20, 29, 110),
    (30, 39, 300),
    (40, 49, 800),
    (50, 64, 3300),
    (65, 79, 21500),
    (80, 150, 150000),
]

In [ ]:
# Set up storage for best solution for each unowned car
best_solutions = {}

# Iterate through solutions
for total_pen, sol in solutions:
    # Set up empty list to store all unowned cars in a given solution
    unowned_in_sol = []
    # Iterate car by car through solution
    for car in sol:
        # If unowned, add to list
        if car[4] > next(u_pen for low, high, u_pen in unowned_values if low <= car[1] <= high):
            unowned_in_sol.append(car)

    # Iterate through all unowned cars in solution
    for car in unowned_in_sol:
        year, rq, mm, v, pen, tune = car
        # If car has solution with lower total penalty in solutions, skip
        # If has solution with equal penalty add
        # If not in solutions, add
        id = f"{year}_[{rq}]_{mm.replace(' ', '_')}"
        if id in best_solutions:
            if total_pen == best_solutions[id]["total_pen"]:
                best_solutions[id]["solutions"].append(sol)
        else:
            best_solutions[id] = {"tuple": car, "total_pen": total_pen, "solutions": [sol]}

In [ ]:
best_solutions

In [ ]:
show_maxed = False

In [ ]:
for id, info in best_solutions.items():
    year, rq, mm, ver, pen, _ = info["tuple"]
    print(f"{year} [{rq}] {mm} ({ver}): {pen} / {info['total_pen']}")
    sol_list = info["solutions"]
    num_sols = len(sol_list)
    for i, sol in enumerate(sol_list):
        if num_sols > 1:
            print(f"Solution {i + 1}")
        for car in sol:
            if show_maxed or pen > 0:
                print("   ", car)

In [ ]:
best_solutions_unique = []
for k, v in best_solutions.items():
    for s in v["solutions"]:
        if s not in best_solutions_unique:
            best_solutions_unique.append(s)

In [ ]:
len(best_solutions_unique)

In [ ]:
best_use_counts = dict()
for sol in best_solutions_unique:
    for year, rq, mm, ver, pen, tune in sol:
        id = f"{year}_[{rq}]_{mm.replace(' ', '_')}_{ver}"
        if id in best_use_counts:
            best_use_counts[id]["count"] += 1
        else:
            best_use_counts[id] = {"tup": (year, rq, mm, ver, pen), "count": 1}

In [ ]:
best_use_counts

In [ ]:
print("Car Uses:")
best_use_counts = dict(
    sorted(best_use_counts.items(), key=lambda item: (-item[1]["count"], item[1]["tup"][4]))
)  # Sort by use count (desc), then pen (asc)
current_uses = 0
for _, car_d in best_use_counts.items():
    car = car_d["tup"]
    uses = car_d["count"]
    if current_uses != uses:
        if uses == len(best_solutions_unique):
            print(f"{uses} Uses (All best solutions):")
        else:
            print(f"{uses} Uses:") if uses != 1 else print(f"{uses} Use:")
        current_uses = uses
    colour = next(c for low, high, c in colour_ranges if low <= car[1] <= high)
    print(f"    {car[0]}\033[48;5;{colour}m[{car[1]}]\033[0m {car[2]} (V{car[3]}): {car[4]}")